# 01 merge harmonized datasets

**Purpose:** Harmonize gene identifiers and merge the four post-QC datasets.

**Original analysis notebook:** `01_merging_core_datasets_fullgene_harmonized_corrected.ipynb`

Run this notebook from top to bottom in a fresh kernel. Project paths are
resolved from `GLIOMA_PROJECT_ROOT`; when that variable is not set, the
notebook searches parent directories for the repository root. Generated
AnnData objects are written under `data/processed/` or `data/interim/`,
while tables, figures, and logs retain the workflow's historical output
directories at the repository root.

# Merge 4 core GBM scRNA-seq datasets — full-gene, gene-symbol harmonized

این نوت‌بوک برای merge اصولی چهار دیتاست postQC استفاده می‌شود:

- DS1 / GSE103224
- DS2 / GSE141383
- DS3 / GSE182109
- DS4 / GSE278450

هدف این نسخه:
1. استفاده از فایل‌های postQC واقعی با حدود 24k تا 30k ژن.
2. حفظ raw counts در `adata.layers["counts"]`.
3. تبدیل Ensembl ID به gene symbol برای DS1 و DS2 با GENCODE v49 GTF.
4. جمع‌کردن duplicate gene symbols، نه حذف‌کردن ساده آن‌ها.
5. merge با ژن‌های مشترک بین چهار دیتاست (`join='inner'`).
6. ذخیره یک full-gene merged object تمیز برای مراحل بعدی مثل HVG/scVI، marker DE و CNV.

**خروجی اصلی:**

`${GLIOMA_PROJECT_ROOT}/processed/integrated/GBM_core_4datasets_fullgene_postQC_merged_counts.h5ad`

In [ ]:
# =========================
# 0) Environment check
# =========================

import sys
from pathlib import Path
import gc
import gzip
import re
from itertools import combinations

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy import sparse

print("Python:", sys.executable)
print("scanpy:", sc.__version__)
print("anndata:", ad.__version__)
print("pandas:", pd.__version__)

In [ ]:
# =========================
# 1) Project paths and configuration
# =========================

import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
RAW_DIR = PROJECT_DIR / "data" / "interim"
REFERENCE_DIR = PROJECT_DIR / "data" / "reference"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
METADATA_DIR = PROJECT_DIR / "metadata"
RESULTS_DIR = PROJECT_DIR / "results"
FIGURES_DIR = PROJECT_DIR / "figures"
LOGS_DIR = PROJECT_DIR / "logs"

INTEGRATED_DIR = PROCESSED_DIR / "integrated"
HARM_DIR = PROCESSED_DIR / "core_4datasets_harmonized_gene_symbols"
COMMON_DIR = HARM_DIR / "common_genes"

for d in [PROCESSED_DIR, METADATA_DIR, RESULTS_DIR, FIGURES_DIR, LOGS_DIR, INTEGRATED_DIR, HARM_DIR, COMMON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# GTF visible in your project screenshots
GTF_PATH = REFERENCE_DIR / "gencode.v49.primary_assembly.annotation.gtf.gz"

# Four postQC input files based on your inspection report
CORE_DATASETS = [
    {
        "core_dataset": "DS1_GSE103224",
        "dataset_id": "DS1",
        "geo_id": "GSE103224",
        "path": PROCESSED_DIR / "DS1_GSE103224" / "GSE103224_postQC.h5ad",
        "gene_id_mode": "ensembl",
    },
    {
        "core_dataset": "DS2_GSE141383",
        "dataset_id": "DS2",
        "geo_id": "GSE141383",
        "path": PROCESSED_DIR / "DS2_GSE141383" / "GSE141383_postQC.h5ad",
        "gene_id_mode": "ensembl",
    },
    {
        "core_dataset": "DS3_GSE182109",
        "dataset_id": "DS3",
        "geo_id": "GSE182109",
        "path": PROCESSED_DIR / "DS3_GSE182109" / "GSE182109_postQC.h5ad",
        "gene_id_mode": "symbol",
    },
    {
        "core_dataset": "DS4_GSE278450",
        "dataset_id": "DS4",
        "geo_id": "GSE278450",
        "path": PROCESSED_DIR / "DS4_GSE278450" / "GSE278450_postQC.h5ad",
        "gene_id_mode": "symbol",
    },
]

# These thresholds are intentionally strict to avoid accidentally merging HVG-only objects.
MIN_GENES_PER_INPUT = 10000
MIN_COMMON_GENES_AFTER_HARMONIZATION = 10000
EXPECTED_FINAL_SHAPE = (351_427, 15_176)

# The four files are already selected as core datasets.
# Keep this False unless you intentionally want to filter rows using existing core-candidate metadata columns.
FILTER_TO_CORE_CANDIDATES_IF_PRESENT = False

# Final output
FINAL_MERGED_H5AD = INTEGRATED_DIR / "GBM_core_4datasets_fullgene_postQC_merged_counts.h5ad"

# Reports
INPUT_INSPECTION_TSV = METADATA_DIR / "core_4datasets_postQC_input_inspection_before_merge_corrected.tsv"
GENE_HARMONIZATION_TSV = METADATA_DIR / "gene_harmonization_report_4datasets_corrected.tsv"
PAIRWISE_INTERSECTION_TSV = METADATA_DIR / "gene_symbol_pairwise_intersections_4datasets_corrected.tsv"
COMMON_GENES_TSV = METADATA_DIR / "common_genes_after_harmonization_4datasets_corrected.tsv"
COMMON_SUBSET_TSV = METADATA_DIR / "common_gene_subset_report_4datasets_corrected.tsv"
MERGE_SUMMARY_TSV = METADATA_DIR / "merged_fullgene_QC_summary_4datasets_corrected.tsv"
CELL_COUNTS_BY_DATASET_TSV = METADATA_DIR / "merged_fullgene_cell_counts_by_dataset_4datasets_corrected.tsv"
CELL_COUNTS_BY_SAMPLE_TSV = METADATA_DIR / "merged_fullgene_cell_counts_by_sample_4datasets_corrected.tsv"
CELL_COUNTS_BY_PATIENT_TSV = METADATA_DIR / "merged_fullgene_cell_counts_by_patient_4datasets_corrected.tsv"

print("PROJECT_DIR:", PROJECT_DIR)
print("GTF_PATH:", GTF_PATH, "| exists:", GTF_PATH.exists())
print("FINAL_MERGED_H5AD:", FINAL_MERGED_H5AD)
print("\nInput files:")
for x in CORE_DATASETS:
    print("-", x["core_dataset"], x["path"], "| exists:", x["path"].exists())

In [ ]:
# =========================
# 2) Safe write / replace helpers
# =========================

def remove_if_exists(path: Path):
    path = Path(path)
    if path.exists():
        path.unlink()
        print(f"[REPLACED old file] {path}")

def write_tsv_replace(df: pd.DataFrame, path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    remove_if_exists(path)
    df.to_csv(path, sep="\t", index=False)
    print(f"[SAVED] {path} shape={df.shape}")

def write_h5ad_replace(adata_obj, path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    remove_if_exists(path)
    adata_obj.write_h5ad(path)
    print(f"[SAVED] {path} shape={adata_obj.shape}")

def clean_string_series(s, fill_value):
    s = pd.Series(s).astype(str).str.strip()
    bad = s.isna() | s.eq("") | s.str.lower().isin(["nan", "none", "null", "na", "n/a"])
    s.loc[bad] = fill_value
    return s

def strip_ensembl_version(values):
    return pd.Series(values).astype(str).str.replace(r"\.\d+$", "", regex=True)

def is_ensembl_like(values, min_fraction=0.5):
    s = pd.Series(values).astype(str)
    if len(s) == 0:
        return False
    return s.str.replace(r"\.\d+$", "", regex=True).str.startswith("ENSG").mean() >= min_fraction

def make_core_mask_if_requested(obs: pd.DataFrame):
    if not FILTER_TO_CORE_CANDIDATES_IF_PRESENT:
        return pd.Series(True, index=obs.index)

    candidate_cols = [
        "strict_core_gbm_idhwt_candidate",
        "core_gbm_idhwt_candidate",
        "is_core",
        "core_candidate",
    ]
    for col in candidate_cols:
        if col in obs.columns:
            s = obs[col].astype(str).str.strip().str.lower()
            return s.isin(["yes", "true", "1", "y", "core"])

    if "analysis_group_major" in obs.columns:
        s = obs["analysis_group_major"].astype(str).str.strip().str.lower()
        return s.eq("core")

    print("[WARNING] Core filtering requested, but no core-candidate column found. Keeping all cells.")
    return pd.Series(True, index=obs.index)

In [ ]:
# =========================
# 3) Build / load GENCODE Ensembl-to-symbol mapping
# =========================

GENCODE_MAP_TSV = METADATA_DIR / "gencode_v49_ensembl_to_gene_symbol_mapping.tsv"

def parse_gtf_attributes(attr_text):
    # GTF attributes look like: gene_id "ENSG..."; gene_name "TP53";
    out = {}
    for item in attr_text.strip().split(";"):
        item = item.strip()
        if not item:
            continue
        if " " not in item:
            continue
        key, val = item.split(" ", 1)
        out[key] = val.strip().strip('"')
    return out

def build_or_load_gencode_map(gtf_path: Path, cache_tsv: Path):
    if cache_tsv.exists():
        m = pd.read_csv(cache_tsv, sep="\t")
        print(f"[USING cached GENCODE map] {cache_tsv} shape={m.shape}")
    else:
        if not gtf_path.exists():
            raise FileNotFoundError(
                f"GENCODE GTF not found: {gtf_path}\n"
                "This file is required to convert DS1/DS2 Ensembl IDs to gene symbols."
            )

        rows = []
        opener = gzip.open if str(gtf_path).endswith(".gz") else open

        with opener(gtf_path, "rt") as fh:
            for line in fh:
                if not line or line.startswith("#"):
                    continue
                parts = line.rstrip("\n").split("\t")
                if len(parts) < 9:
                    continue
                feature_type = parts[2]
                if feature_type != "gene":
                    continue

                chrom, source, feature, start, end, score, strand, frame, attrs = parts
                d = parse_gtf_attributes(attrs)

                gene_id = d.get("gene_id", "")
                gene_name = d.get("gene_name", "")
                gene_type = d.get("gene_type", d.get("gene_biotype", ""))

                if not gene_id or not gene_name:
                    continue

                rows.append({
                    "ensembl_id_raw": gene_id,
                    "ensembl_id": re.sub(r"\.\d+$", "", gene_id),
                    "gene_symbol": gene_name,
                    "gene_type": gene_type,
                    "chromosome": chrom,
                    "start": int(start),
                    "end": int(end),
                    "strand": strand,
                })

        m = pd.DataFrame(rows).drop_duplicates(subset=["ensembl_id"], keep="first")
        write_tsv_replace(m, cache_tsv)
        print(f"[BUILT GENCODE map] {cache_tsv} shape={m.shape}")

    # dict: ENSG without version -> symbol
    map_dict = dict(zip(m["ensembl_id"].astype(str), m["gene_symbol"].astype(str)))
    return m, map_dict

gencode_map, ensembl_to_symbol = build_or_load_gencode_map(GTF_PATH, GENCODE_MAP_TSV)

print("GENCODE mapping rows:", len(gencode_map))
display(gencode_map.head())

In [ ]:
# =========================
# 4) Inspect the four postQC inputs
# =========================

inspection_rows = []

for item in CORE_DATASETS:
    p = Path(item["path"])
    row = {
        "core_dataset": item["core_dataset"],
        "dataset_id_config": item["dataset_id"],
        "geo_id_config": item["geo_id"],
        "path": str(p),
        "exists": p.exists(),
    }

    print("\n" + "=" * 100)
    print(item["core_dataset"])
    print(p)
    print("exists:", p.exists())

    if not p.exists():
        inspection_rows.append(row)
        continue

    a = sc.read_h5ad(p)

    row.update({
        "n_obs": int(a.n_obs),
        "n_vars": int(a.n_vars),
        "layers": ",".join(list(a.layers.keys())),
        "has_counts_layer": "counts" in a.layers,
        "n_obs_columns": len(a.obs.columns),
        "n_var_columns": len(a.var.columns),
        "duplicated_var_names": int(a.var_names.duplicated().sum()),
        "first_10_var_names": ";".join(map(str, a.var_names[:10])),
        "var_names_ensembl_like": is_ensembl_like(a.var_names),
        "obs_columns": ";".join(map(str, a.obs.columns)),
        "var_columns": ";".join(map(str, a.var.columns)),
    })

    print(a)
    print("layers:", list(a.layers.keys()))
    print("first 10 var_names:", list(a.var_names[:10]))
    print("var_names Ensembl-like:", row["var_names_ensembl_like"])

    if "counts" not in a.layers:
        raise ValueError(f"{p} is missing adata.layers['counts']. Stop here and fix postQC object.")

    if a.n_vars < MIN_GENES_PER_INPUT:
        raise ValueError(
            f"{p} has only {a.n_vars} genes. This looks like HVG/subset object, not full-gene postQC."
        )

    del a
    gc.collect()

    inspection_rows.append(row)

input_inspection = pd.DataFrame(inspection_rows)
write_tsv_replace(input_inspection, INPUT_INSPECTION_TSV)
display(input_inspection)

In [ ]:
# =========================
# 5) Gene symbol selection and duplicate-collapse helpers
# =========================

def choose_gene_symbols(adata_obj, item, ensembl_to_symbol):
    '''
    Returns:
        symbols: pd.Series length n_vars, index original var_names
        source: text describing source
        mapped_fraction: fraction of Ensembl IDs mapped when Ensembl mode is used, otherwise NaN
    '''
    original_var_names = pd.Series(adata_obj.var_names.astype(str), index=adata_obj.var_names.astype(str))

    # First try configured mode.
    mode = item.get("gene_id_mode", "auto")

    if mode == "ensembl" or (mode == "auto" and is_ensembl_like(original_var_names)):
        ensg = strip_ensembl_version(original_var_names.values)
        symbols = ensg.map(ensembl_to_symbol)
        mapped_fraction = float(symbols.notna().mean())

        # Fallback to obvious var columns if mapping unexpectedly poor.
        if mapped_fraction < 0.50:
            candidate_symbol_cols = [
                "gene_symbol", "gene_symbols", "feature_name", "gene_name", "symbol"
            ]
            for col in candidate_symbol_cols:
                if col in adata_obj.var.columns:
                    fallback = adata_obj.var[col].astype(str).str.strip()
                    if fallback.notna().mean() > 0.50:
                        print(f"[WARNING] GTF mapping low ({mapped_fraction:.3f}); using var['{col}'] fallback.")
                        return pd.Series(fallback.values, index=adata_obj.var_names), f"var[{col}]_fallback", mapped_fraction

        return pd.Series(symbols.values, index=adata_obj.var_names), "gencode_v49_ensembl_to_gene_symbol", mapped_fraction

    # Gene symbol mode
    candidate_symbol_cols = [
        "gene_symbol", "gene_symbols", "feature_name", "gene_name", "symbol"
    ]

    # Prefer var_names if they are not Ensembl-like; this matches DS3 and DS4 inspection.
    if not is_ensembl_like(original_var_names):
        return pd.Series(original_var_names.values, index=adata_obj.var_names), "var_names_gene_symbol", np.nan

    for col in candidate_symbol_cols:
        if col in adata_obj.var.columns:
            return pd.Series(adata_obj.var[col].astype(str).values, index=adata_obj.var_names), f"var[{col}]", np.nan

    return pd.Series(original_var_names.values, index=adata_obj.var_names), "var_names", np.nan


def standardize_obs_metadata(adata_obj, item):
    obs = adata_obj.obs.copy()

    core_dataset = item["core_dataset"]
    dataset_id = item["dataset_id"]
    geo_id = item["geo_id"]

    obs["core_dataset"] = core_dataset
    obs["dataset_id"] = dataset_id
    obs["geo_id"] = geo_id
    obs["study_id"] = geo_id
    obs["source_dataset"] = core_dataset
    obs["source_file"] = str(item["path"])

    if "sample_id" not in obs.columns:
        # Try common alternatives, otherwise use dataset-level sample.
        for col in ["sample", "sample_name", "orig.ident", "library_id"]:
            if col in obs.columns:
                obs["sample_id"] = obs[col].astype(str)
                break
        else:
            obs["sample_id"] = f"{core_dataset}_sample_unknown"
    obs["sample_id"] = clean_string_series(obs["sample_id"], f"{core_dataset}_sample_unknown").values

    if "patient_id" not in obs.columns:
        for col in ["patient", "patient_name", "donor", "donor_id", "case_id"]:
            if col in obs.columns:
                obs["patient_id"] = obs[col].astype(str)
                break
        else:
            # If patient is not available, sample_id is safer than blank.
            obs["patient_id"] = obs["sample_id"].astype(str)
    obs["patient_id"] = clean_string_series(obs["patient_id"], f"{core_dataset}_patient_unknown").values

    if "condition" not in obs.columns:
        for col in ["diagnosis", "disease", "group"]:
            if col in obs.columns:
                obs["condition"] = obs[col].astype(str)
                break
        else:
            obs["condition"] = "GBM"
    obs["condition"] = clean_string_series(obs["condition"], "GBM").values

    # Useful batch columns for downstream scVI.
    obs["batch_dataset"] = obs["core_dataset"].astype(str)
    obs["batch_sample"] = obs["sample_id"].astype(str)
    obs["batch_patient"] = obs["patient_id"].astype(str)

    # Preserve original barcode before prefixing.
    obs["original_cell_barcode"] = adata_obj.obs_names.astype(str)

    return obs


def collapse_to_gene_symbols_counts_only(adata_obj, symbols, item):
    '''
    Build a new AnnData with:
      X = raw counts
      layers['counts'] = raw counts
      var_names = unique gene symbols
    Duplicate gene symbols are summed using sparse matrix multiplication.
    '''
    obs = standardize_obs_metadata(adata_obj, item)

    symbols = pd.Series(symbols).astype(str).str.strip()
    invalid = (
        symbols.isna()
        | symbols.eq("")
        | symbols.str.lower().isin(["nan", "none", "null", "na", "n/a"])
        | symbols.str.startswith("ENSG")  # after mapping, remaining ENSG are unmapped and should not be mixed with symbols
    )

    valid_mask = (~invalid).to_numpy()
    n_invalid = int(invalid.sum())

    original_var_names = pd.Series(adata_obj.var_names.astype(str))
    original_var_names_valid = original_var_names.loc[valid_mask].reset_index(drop=True)
    symbols_valid = symbols.loc[valid_mask].reset_index(drop=True)

    counts = adata_obj.layers["counts"]
    if sparse.issparse(counts):
        counts = counts.tocsr()[:, valid_mask]
    else:
        counts = sparse.csr_matrix(counts[:, valid_mask])

    group_ids, unique_symbols = pd.factorize(symbols_valid, sort=False)
    n_unique = len(unique_symbols)

    # Sparse column aggregation matrix: original genes -> unique gene symbols
    data = np.ones(len(group_ids), dtype=counts.dtype)
    rows = np.arange(len(group_ids))
    cols = group_ids
    G = sparse.csr_matrix((data, (rows, cols)), shape=(len(group_ids), n_unique))

    counts_agg = (counts @ G).tocsr()

    # Gene-level metadata after collapsing
    collapse_df = pd.DataFrame({
        "gene_symbol": symbols_valid.astype(str),
        "original_var_name": original_var_names_valid.astype(str),
    })

    var = (
        collapse_df
        .groupby("gene_symbol", sort=False)
        .agg(
            n_original_features_collapsed=("original_var_name", "size"),
            original_var_names_joined=("original_var_name", lambda x: ";".join(map(str, x[:20])) + (";..." if len(x) > 20 else "")),
        )
        .reset_index()
    )
    var.index = pd.Index(var["gene_symbol"].astype(str), name=None)
    var = var.drop(columns=["gene_symbol"])

    new = ad.AnnData(
        X=counts_agg.copy(),
        obs=obs,
        var=var,
    )
    new.layers["counts"] = counts_agg.copy()

    # Prefix cell names after constructing obs.
    new.obs_names = [f"{item['core_dataset']}:{x}" for x in adata_obj.obs_names.astype(str)]
    new.obs_names_make_unique()
    new.var_names_make_unique()

    report = {
        "core_dataset": item["core_dataset"],
        "input_file": str(item["path"]),
        "n_cells_input": int(adata_obj.n_obs),
        "n_genes_input": int(adata_obj.n_vars),
        "n_invalid_or_unmapped_gene_symbols_removed": n_invalid,
        "n_features_after_invalid_removal": int(valid_mask.sum()),
        "n_unique_gene_symbols_after_collapse": int(new.n_vars),
        "n_duplicate_features_collapsed": int(valid_mask.sum() - new.n_vars),
        "has_counts_layer_output": "counts" in new.layers,
    }

    return new, report

In [ ]:
# =========================
# 6) Harmonize each dataset to gene symbols
# =========================

# Replace only outputs produced by this notebook.
for p in [
    INPUT_INSPECTION_TSV,
    GENE_HARMONIZATION_TSV,
    PAIRWISE_INTERSECTION_TSV,
    COMMON_GENES_TSV,
    COMMON_SUBSET_TSV,
    MERGE_SUMMARY_TSV,
    CELL_COUNTS_BY_DATASET_TSV,
    CELL_COUNTS_BY_SAMPLE_TSV,
    CELL_COUNTS_BY_PATIENT_TSV,
    FINAL_MERGED_H5AD,
]:
    if Path(p).exists():
        remove_if_exists(Path(p))

# Clean intermediate h5ad files in this notebook's harmonization folder only.
for p in sorted(HARM_DIR.glob("*.h5ad")) + sorted(COMMON_DIR.glob("*.h5ad")):
    remove_if_exists(p)

harmonized_files = {}
gene_sets = {}
harmonization_rows = []

for item in CORE_DATASETS:
    print("\n" + "=" * 100)
    print("[READ]", item["core_dataset"], item["path"])

    if not Path(item["path"]).exists():
        raise FileNotFoundError(item["path"])

    a = sc.read_h5ad(item["path"])
    print("Input:", a)

    if "counts" not in a.layers:
        raise ValueError(f"{item['path']} does not contain adata.layers['counts'].")

    if a.n_vars < MIN_GENES_PER_INPUT:
        raise ValueError(
            f"{item['path']} has only {a.n_vars} genes. "
            "This is not acceptable for full-gene merge."
        )

    core_mask = make_core_mask_if_requested(a.obs)
    if int(core_mask.sum()) == 0:
        raise ValueError(f"No cells retained after optional core filtering: {item['core_dataset']}")

    if int(core_mask.sum()) < a.n_obs:
        print(f"[FILTER] keeping {int(core_mask.sum())} / {a.n_obs} cells by core-candidate metadata")
        a = a[core_mask.to_numpy()].copy()

    symbols, symbol_source, mapped_fraction = choose_gene_symbols(a, item, ensembl_to_symbol)

    print("symbol_source:", symbol_source)
    print("mapped_fraction:", mapped_fraction)

    h, rep = collapse_to_gene_symbols_counts_only(a, symbols, item)
    rep.update({
        "symbol_source": symbol_source,
        "ensembl_mapped_fraction": mapped_fraction,
        "n_samples": int(h.obs["sample_id"].nunique()),
        "n_patients": int(h.obs["patient_id"].nunique()),
        "conditions": " | ".join(sorted(h.obs["condition"].astype(str).unique())),
    })

    if h.n_vars < MIN_GENES_PER_INPUT:
        raise ValueError(
            f"After harmonization, {item['core_dataset']} has only {h.n_vars} gene symbols. "
            "This indicates failed Ensembl-to-symbol mapping or wrong input."
        )

    out_h5ad = HARM_DIR / f"{item['core_dataset']}_postQC_gene_symbol_harmonized_counts.h5ad"
    write_h5ad_replace(h, out_h5ad)

    harmonized_files[item["core_dataset"]] = out_h5ad
    gene_sets[item["core_dataset"]] = set(map(str, h.var_names))
    rep["harmonized_h5ad"] = str(out_h5ad)
    harmonization_rows.append(rep)

    print("Output:", h)

    del a, h
    gc.collect()

gene_harmonization = pd.DataFrame(harmonization_rows)
write_tsv_replace(gene_harmonization, GENE_HARMONIZATION_TSV)
display(gene_harmonization)

In [ ]:
# =========================
# 7) Compute shared genes across the 4 harmonized datasets
# =========================

pair_rows = []
dataset_keys = [x["core_dataset"] for x in CORE_DATASETS]

for k1, k2 in combinations(dataset_keys, 2):
    pair_rows.append({
        "dataset_1": k1,
        "dataset_2": k2,
        "n_genes_1": len(gene_sets[k1]),
        "n_genes_2": len(gene_sets[k2]),
        "pairwise_intersection": len(gene_sets[k1].intersection(gene_sets[k2])),
    })

pairwise = pd.DataFrame(pair_rows)

common_genes = sorted(set.intersection(*[gene_sets[k] for k in dataset_keys]))
common_genes_df = pd.DataFrame({"gene_symbol": common_genes})

write_tsv_replace(pairwise, PAIRWISE_INTERSECTION_TSV)
write_tsv_replace(common_genes_df, COMMON_GENES_TSV)

print("Common gene symbols across all 4 datasets:", len(common_genes))
display(pairwise)

if len(common_genes) < MIN_COMMON_GENES_AFTER_HARMONIZATION:
    raise RuntimeError(
        f"Too few common genes after harmonization: {len(common_genes)}. "
        "Do not continue. Check gene mapping and input files."
    )

In [ ]:
# =========================
# 8) Subset each harmonized dataset to common genes
# =========================

common_files = {}
subset_rows = []

for item in CORE_DATASETS:
    key = item["core_dataset"]
    print("\n" + "=" * 100)
    print("[COMMON SUBSET]", key)

    h = sc.read_h5ad(harmonized_files[key])

    # Keep the same common gene order for all datasets
    missing = sorted(set(common_genes) - set(map(str, h.var_names)))
    if missing:
        raise ValueError(f"{key} is missing {len(missing)} common genes unexpectedly.")

    h = h[:, common_genes].copy()

    # Ensure X and counts are raw counts and aligned
    if "counts" not in h.layers:
        raise ValueError(f"{key} lost counts layer before common-gene subset.")
    h.X = h.layers["counts"].copy()

    out_common = COMMON_DIR / f"{key}_postQC_common_genes_counts.h5ad"
    write_h5ad_replace(h, out_common)

    common_files[key] = out_common
    subset_rows.append({
        "core_dataset": key,
        "common_subset_h5ad": str(out_common),
        "n_cells": int(h.n_obs),
        "n_common_genes": int(h.n_vars),
        "has_counts_layer": "counts" in h.layers,
        "first_10_common_genes": ";".join(map(str, h.var_names[:10])),
    })

    print("Subset output:", h)

    del h
    gc.collect()

common_subset_report = pd.DataFrame(subset_rows)
write_tsv_replace(common_subset_report, COMMON_SUBSET_TSV)
display(common_subset_report)

In [ ]:
# =========================
# 9) Concatenate / merge the four common-gene objects
# =========================

print("[FINAL OUTPUT]", FINAL_MERGED_H5AD)
remove_if_exists(FINAL_MERGED_H5AD)

concat_method = None

# Prefer concat_on_disk for memory safety.
try:
    print("[TRY] anndata.experimental.concat_on_disk")
    ad.experimental.concat_on_disk(
        in_files={key: str(common_files[key]) for key in dataset_keys},
        out_file=str(FINAL_MERGED_H5AD),
        axis=0,
        join="inner",
        label="concat_key",
        index_unique=None,
        merge="same",
        uns_merge="unique",
    )
    concat_method = "anndata.experimental.concat_on_disk"

except Exception as e:
    print("[WARNING] concat_on_disk failed.")
    print("Reason:", repr(e))
    print("[FALLBACK] in-memory anndata.concat")

    adatas = {}
    for key in dataset_keys:
        print("Loading:", key, common_files[key])
        adatas[key] = sc.read_h5ad(common_files[key])
        if "counts" not in adatas[key].layers:
            raise ValueError(f"{key} has no counts layer before concat.")

    merged = ad.concat(
        adatas,
        axis=0,
        join="inner",
        label="concat_key",
        index_unique=None,
        merge="same",
        uns_merge="unique",
    )

    merged.obs_names_make_unique()

    if "counts" not in merged.layers:
        raise ValueError("After concat, merged.layers['counts'] is missing.")

    merged.X = merged.layers["counts"].copy()

    merged.uns["core_4datasets_merge_info"] = {
        "description": "Full-gene postQC merge of four selected GBM scRNA-seq datasets; gene symbols harmonized; duplicate gene symbols collapsed by summing raw counts; common genes only.",
        "datasets": dataset_keys,
        "join": "inner_common_genes",
        "raw_counts": "stored in X and layers['counts']",
        "gene_harmonization": "DS1/DS2 Ensembl mapped with GENCODE v49; DS3/DS4 gene symbols from var_names",
        "n_cells": int(merged.n_obs),
        "n_genes": int(merged.n_vars),
    }

    write_h5ad_replace(merged, FINAL_MERGED_H5AD)
    concat_method = "in_memory_anndata_concat"

    del merged, adatas
    gc.collect()

print("[MERGED]", FINAL_MERGED_H5AD, "method:", concat_method)

In [ ]:
# =========================
# 10) Final QC and reports
# =========================

adata_final = sc.read_h5ad(FINAL_MERGED_H5AD, backed="r")

print("Final merged object:")
print(adata_final)

if adata_final.shape != EXPECTED_FINAL_SHAPE:
    raise AssertionError(
        f"Expected final merged shape {EXPECTED_FINAL_SHAPE}, observed {adata_final.shape}."
    )

if adata_final.n_vars < MIN_COMMON_GENES_AFTER_HARMONIZATION:
    raise ValueError(
        f"Final merged object has only {adata_final.n_vars} genes. "
        "This is too low for full-gene downstream analysis."
    )

has_counts = "counts" in adata_final.layers
if not has_counts:
    raise ValueError("Final merged object has no layers['counts'].")

obs = adata_final.obs.copy()

merge_summary = pd.DataFrame([{
    "final_h5ad": str(FINAL_MERGED_H5AD),
    "concat_method": concat_method,
    "n_cells": int(adata_final.n_obs),
    "n_genes": int(adata_final.n_vars),
    "has_counts_layer": has_counts,
    "n_core_datasets": int(obs["core_dataset"].nunique()) if "core_dataset" in obs.columns else None,
    "n_samples": int(obs["sample_id"].nunique()) if "sample_id" in obs.columns else None,
    "n_patients": int(obs["patient_id"].nunique()) if "patient_id" in obs.columns else None,
    "first_10_genes": ";".join(map(str, adata_final.var_names[:10])),
    "last_10_genes": ";".join(map(str, adata_final.var_names[-10:])),
    "note": "This is the clean full-gene/common-gene postQC merged object. Use it as the base object; create HVG/scVI branch from this object, not the reverse.",
}])

write_tsv_replace(merge_summary, MERGE_SUMMARY_TSV)

cell_counts_by_dataset = (
    obs.groupby(["core_dataset", "dataset_id", "geo_id"], observed=True)
    .size()
    .reset_index(name="n_cells")
    .sort_values("n_cells", ascending=False)
)
write_tsv_replace(cell_counts_by_dataset, CELL_COUNTS_BY_DATASET_TSV)

if "sample_id" in obs.columns:
    cell_counts_by_sample = (
        obs.groupby(["core_dataset", "sample_id"], observed=True)
        .size()
        .reset_index(name="n_cells")
        .sort_values(["core_dataset", "n_cells"], ascending=[True, False])
    )
    write_tsv_replace(cell_counts_by_sample, CELL_COUNTS_BY_SAMPLE_TSV)
else:
    cell_counts_by_sample = pd.DataFrame()

if "patient_id" in obs.columns:
    cell_counts_by_patient = (
        obs.groupby(["core_dataset", "patient_id"], observed=True)
        .size()
        .reset_index(name="n_cells")
        .sort_values(["core_dataset", "n_cells"], ascending=[True, False])
    )
    write_tsv_replace(cell_counts_by_patient, CELL_COUNTS_BY_PATIENT_TSV)
else:
    cell_counts_by_patient = pd.DataFrame()

display(merge_summary)
display(cell_counts_by_dataset)

adata_final.file.close()

In [ ]:
# =========================
# 11) Sanity check: reopen final object normally
# =========================

# This cell loads the final merged h5ad in memory only to confirm it can be read.
# If memory is tight, you may skip this cell after the backed QC above succeeded.

adata_check = sc.read_h5ad(FINAL_MERGED_H5AD)
print(adata_check)
print("counts layer:", "counts" in adata_check.layers)
print("X shape:", adata_check.X.shape)
print("counts shape:", adata_check.layers["counts"].shape if "counts" in adata_check.layers else None)
print("n genes:", adata_check.n_vars)
print("n cells:", adata_check.n_obs)
print("obs columns preview:", list(adata_check.obs.columns)[:40])
print("first 20 genes:", list(adata_check.var_names[:20]))

if adata_check.n_vars < MIN_COMMON_GENES_AFTER_HARMONIZATION:
    raise ValueError("Final object is not full-gene enough. Stop before HVG/scVI.")

if "counts" not in adata_check.layers:
    raise ValueError("counts layer missing after reopen.")

del adata_check
gc.collect()

print("All merge checks passed.")

## Files to send back after running this notebook

بعد از اجرای کامل، این فایل‌ها را برای بررسی بفرست:

1. `gene_harmonization_report_4datasets_corrected.tsv`
2. `gene_symbol_pairwise_intersections_4datasets_corrected.tsv`
3. `common_genes_after_harmonization_4datasets_corrected.tsv`
4. `common_gene_subset_report_4datasets_corrected.tsv`
5. `merged_fullgene_QC_summary_4datasets_corrected.tsv`
6. `merged_fullgene_cell_counts_by_dataset_4datasets_corrected.tsv`

فایل h5ad نهایی احتمالاً بزرگ است و لازم نیست بفرستی، مگر اینکه خطا یا mismatch داشته باشیم.